# Making Batches of Data to a Folder

Note that this will save data both into our lab-wide Tiled server and into our local folder

Here we use the ophyd-async devices

You'll need to have TILED_API_KEY and TILED_URI environment variables set.

## Imports

### Global Imports

In [1]:
import numpy as np

from bluesky import RunEngine
from bluesky.callbacks import LiveTable
from bluesky.plans import count, list_scan
import bluesky.plan_stubs as bps

from bluesky.callbacks.tiled_writer import TiledWriter
from tiled.server import SimpleTiledServer
from tiled.client import from_uri

from ophyd_async.core import init_devices

import matplotlib.pyplot as plt

### Local-File Imports

In [2]:
from sidekick_model3_CA_devices_v4 import PulseGenerator, LilLaser, Diode
from sidekick_model3_PVA_devices_v4 import DiodePVA
from array_steps import array_one_nd_step_with_reps
from sidekick_model3_helpers import condition_pulse

## Initialize our Sidekick Model 3 Devices

#### Start Bluesky RunEngine (before initializing ophyd-async devices)

In [3]:
RE = RunEngine()

#### Initialize ophyd-async devices

In [4]:
with init_devices():
    pulsegen = PulseGenerator("PULSEGEN:", name="pulsegen")
    laser = LilLaser("LASER:", name="laser")
    electron = Diode("ELECTRON:", name="electron")
    proton = Diode("PROTON:", name="proton")
    electron_pva = DiodePVA(prefix="pva://ELECTRON-DAQ:", name="electron_pva")
    proton_pva = DiodePVA(prefix="pva://PROTON-DAQ:", name="proton_pva")

## Connect to our Lab-Wide Tiled Server, and Subscribe to Run Engine

In [5]:
import os
from tiled.client import from_uri

def get_tiled_client():
    """Connect to Tiled using environment variables.
    Simple helper function written by ChatGPT"""

    tiled_uri = os.environ.get("TILED_URI")
    tiled_api_key = os.environ.get("TILED_API_KEY")

    if tiled_uri is None:
        raise RuntimeError(
            "Missing TILED_URI.\n"
            "Example:\n"
            "    export TILED_URI='http://your-tiled-server.lan:8000'\n"
        )

    if tiled_api_key is None:
        raise RuntimeError(
            "Missing TILED_API_KEY.\n"
            "Example:\n"
            "    export TILED_API_KEY='your-api-key-here'\n"
        )

    return from_uri(tiled_uri, api_key=tiled_api_key)

In [6]:
tiled_client = get_tiled_client()

In [7]:
tw = TiledWriter(tiled_client)
RE.subscribe(tw)

0

## Create and Execute Plan to Put Sidekick in a Known Initial State

### Create Plan to Set timing settings and rep-rate.

In [22]:
# Written by ChatGPT with help from Scott Feister on 2026-06-22.
# Simple Bluesky pre-run setup helper for putting Sidekick devices
# into a known initial state before opening a run.

def prepare_for_run():
    """Put the Sidekick Model 3 into the standard initial state before a run.
    """

    # Set all delta-t values.
    yield from bps.mv(
        proton.dt, 5.0e-6,        # seconds
        electron.dt, 5.0e-6,      # seconds
        laser.powers_dt, 5.0,     # microseconds
    )

    # Set trigger delays to known values.
    yield from bps.mv(
        pulsegen.ch2_delay, 100.0,    # microseconds; proton delay
        pulsegen.ch3_delay, 100.0,    # microseconds; electron delay
        pulsegen.ch4_delay, 100.0,    # microseconds; laser delay
    )

    # Set system repetition rate.
    yield from bps.mv(
        pulsegen.reprate, 10.0,       # Hz
    )

### Execute this Plan

In [23]:
RE(prepare_for_run())

()

## One-off Test: 50 random traces with 3 reps on each?
Empirically, it takes about 130 ms per laser setting (one rep) or 140 ms per laser setting (two reps) or 195 ms per laser setting (three reps) or 760 ms per laser (twenty reps) or 1400 ms per laser setting (40 reps). So like 35 ms per rep, and 100 ms per laser setting. Rep rate is 50 Hz, or 20 ms per trace at best.

Anyway, empirically 200 random traces with 3 reps each takes about 40 seconds. This makes it a decent batch size; long enough to make the Tiled connection overhead insignificant, short enough to feel "real-time".

In [24]:
npulses = 50
reps = 3

pulse_list = [condition_pulse(np.round(np.random.rand(100)*255)) for i in range(npulses)]
pulse_list[0].shape

(100,)

In [25]:
%%time

detectors = [electron_pva.trace, proton_pva.trace] # note that these both have "trigger()" methods that force a new uniqueId before reading
motor = laser.powers
motor_points = pulse_list
md = {"user_note" : "A bunch of random traces, list scan on Sidekick Model 3"}
uid, = RE(list_scan(detectors, motor, motor_points, md=md, per_step=array_one_nd_step_with_reps(reps=reps)))

CPU times: total: 4.61 s
Wall time: 28.7 s


### Export data

In [26]:
"""
Export one specific kind of run to from Tiled to HDF5.

Created by ChatGPT with help from Scott Feister on 2026-06-25.
"""

from pathlib import Path

import h5py
import numpy as np


def export_run_to_hdf5_custom2(tiled_client, uid, filename=None):
    if filename is None:
        filename = f"run_{uid}.h5"

    filename = Path(filename)
    filename.parent.mkdir(parents=True, exist_ok=True)

    primary = tiled_client[uid, "primary"]

    with h5py.File(filename, "w") as f:
        f.attrs["uid"] = uid

        g = f.create_group("primary")

        g.create_dataset("scan_step", data=np.asarray(primary["scan_step"]))
        g.create_dataset("scan_rep", data=np.asarray(primary["scan_rep"]))
        g.create_dataset("laser_powers", data=np.asarray(primary["laser-powers-readback"]))
        g.create_dataset("electron_trace", data=np.asarray(primary["electron_pva-trace-array"]))
        g.create_dataset("electron_shot_num", data=np.asarray(primary["electron_pva-trace-uniqueId"]))
        g.create_dataset("proton_trace", data=np.asarray(primary["proton_pva-trace-array"]))
        g.create_dataset("proton_shot_num", data=np.asarray(primary["proton_pva-trace-uniqueId"]))

    return filename

In [27]:
primary = tiled_client[uid, "primary"]

In [28]:
primary["laser-powers-readback"].shape

(150, 100)

In [29]:
export_run_to_hdf5_custom2(tiled_client, uid, f"outputs/run_{uid}.h5")

WindowsPath('outputs/run_f65f0c7e-d8ad-4ea1-9ccc-9466735fe094.h5')

In [20]:
#!/usr/bin/env python3
"""
export_batch_hdf5.py

Export one Tiled run to the next numbered HDF5 batch file.

Each exported run becomes one file:

    outputs/batches/batch_001.h5
    outputs/batches/batch_002.h5
    ...

Written by ChatGPT with help from Scott Feister on 2026-06-26.
"""

from pathlib import Path
import re


# -----------------------
# Settings
# -----------------------

outputs_dir = Path("outputs")
batches_dir = outputs_dir / "batches"


# -----------------------
# Helper: find next batch number
# -----------------------

def get_existing_batch_numbers():
    batch_numbers = []

    for path in batches_dir.glob("batch_*"):
        # Accept both old-style folders:
        #     batch_001/
        #
        # and new-style HDF5 files:
        #     batch_001.h5
        match = re.fullmatch(r"batch_(\d+)(?:\.h5)?", path.name)

        if match:
            batch_numbers.append(int(match.group(1)))

    return sorted(batch_numbers)


def get_next_batch_number():
    existing_numbers = get_existing_batch_numbers()

    if not existing_numbers:
        return 1

    return max(existing_numbers) + 1


def get_next_batch_hdf5_path():
    batches_dir.mkdir(parents=True, exist_ok=True)

    batch_number = get_next_batch_number()
    batch_name = f"batch_{batch_number:03d}.h5"

    return batches_dir / batch_name


# -----------------------
# Export one run
# -----------------------

def export_run_as_next_batch(tiled_client, uid):
    batch_path = get_next_batch_hdf5_path()

    export_run_to_hdf5_custom2(
        tiled_client,
        uid,
        batch_path,
    )

    print(f"Wrote {batch_path}")

    return batch_path

In [21]:
export_run_as_next_batch(tiled_client, uid)

Wrote outputs\batches\batch_017.h5


WindowsPath('outputs/batches/batch_017.h5')

## Run and Save - in a Loop

Note that this has failed before because of a single rogue waveform from laser_powers coming into Tiled as 9 elements long.

In [17]:
detectors = [electron_pva.trace, proton_pva.trace]
motor = laser.powers

for i in range(20):
    pulse_list = [condition_pulse(np.round(np.random.rand(100)*255)) for i in range(npulses)]
    motor_points = pulse_list
    md = {"user_note" : "Acquire and save in a loop, repeated list scans on Sidekick Model 3", "outer_loop_i": i}
    uid, = RE(list_scan(detectors, motor, motor_points, md=md, per_step=array_one_nd_step_with_reps(reps=reps)))
    export_run_as_next_batch(tiled_client, uid)

Wrote outputs\batches\batch_015.h5


TypeError: Object dtype dtype('O') has no native HDF5 equivalent

In [17]:
primary = tiled_client[uid]["primary"]

In [18]:
primary

<BlueskyEventStream {'electron_pva-trace-array', 'scan_rep', 'proton_pva-trace-uniqueId', 'electron_pva-trace-timeStamp', 'electron_pva-trace-uniqueId', 'time', 'proton_pva-trace-array', 'scan_step', 'proton_pva-trace-timeStamp', 'laser-powers-readback'} stream_name='primary'>

In [20]:
primary["laser-powers-readback"]

<RaggedClient shape=(600, None) size=59945 chunks=((600,), None) dtype=int64>

In [22]:
laser_powers = primary["laser-powers-readback"].read()

bad = []

for i, x in enumerate(laser_powers):
    arr = np.asarray(x)
    if len(arr) != 100:
        bad.append((i, len(arr)))

print(bad[:50])
print(f"Number of bad rows: {len(bad)}")


lengths = [len(np.asarray(x)) for x in laser_powers]

print(len(lengths))
print(min(lengths))
print(max(lengths))
print(sorted(set(lengths))[:20])
print(sum(lengths))

[(88, 45)]
Number of bad rows: 1
600
45
100
[45, 100]
59945


In [23]:
for i, x in enumerate(laser_powers):
    arr = np.asarray(x)
    if len(arr) != 100:
        print(i, arr.shape, arr)

88 (45,) [ragged.array(140) ragged.array(1) ragged.array(33) ragged.array(172)
 ragged.array(48) ragged.array(81) ragged.array(200) ragged.array(41)
 ragged.array(40) ragged.array(219) ragged.array(101) ragged.array(49)
 ragged.array(131) ragged.array(141) ragged.array(152) ragged.array(6)
 ragged.array(129) ragged.array(34) ragged.array(249) ragged.array(106)
 ragged.array(102) ragged.array(139) ragged.array(158) ragged.array(47)
 ragged.array(166) ragged.array(253) ragged.array(189) ragged.array(47)
 ragged.array(193) ragged.array(150) ragged.array(180) ragged.array(80)
 ragged.array(32) ragged.array(62) ragged.array(127) ragged.array(126)
 ragged.array(223) ragged.array(98) ragged.array(147) ragged.array(51)
 ragged.array(235) ragged.array(191) ragged.array(105) ragged.array(154)
 ragged.array(52)]


In [24]:
bad_i = 88  # replace this with the bad index

print("scan_step:", np.asarray(primary["scan_step"])[bad_i])
print("scan_rep:", np.asarray(primary["scan_rep"])[bad_i])

print("laser powers shape:", np.asarray(primary["laser-powers-readback"][bad_i]).shape)

print("electron trace shape:", np.asarray(primary["electron_pva-trace-array"][bad_i]).shape)
print("proton trace shape:", np.asarray(primary["proton_pva-trace-array"][bad_i]).shape)

print("electron shot:", np.asarray(primary["electron_pva-trace-uniqueId"])[bad_i])
print("proton shot:", np.asarray(primary["proton_pva-trace-uniqueId"])[bad_i])

scan_step: 29
scan_rep: 1
laser powers shape: (45,)
electron trace shape: (100,)
proton trace shape: (100,)
electron shot: 13580268
proton shot: 13580269


In [63]:
uid

'007d34f3-9a1a-4aa7-b57f-9b3f099fe73d'

In [32]:
pulsegen.describe

<bound method StandardReadable.describe of <sidekick_model3_CA_devices_v4.PulseGenerator object at 0x00000281A7A3C2D0>>